# city_selection_and_coverage_audit
# This notebook performs a coverage audit of candidate cities for the air quality project, focusing on PM2.5 and PM10 data availability. It uses the OpenAQ API to query locations and measurements, compiling results into a comprehensive table for review. Later we will use these location IDs to fetch data from aws S3

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
import requests

API key from .env file

In [ ]:
BASE_URL = "https://api.openaq.org/v3"

# OpenAQ API key configuration
OPENAQ_API_KEY = os.getenv("OPENAQ_API_KEY")

if OPENAQ_API_KEY is None:
    raise RuntimeError(
        "OPENAQ_API_KEY is missing. The key can be stored as an environment variable before running this notebook."
    )

HEADERS = {"X-API-Key": OPENAQ_API_KEY}

### Section 2: Gulf city bounding boxes

The bounding boxes use this order:

longitude_min, latitude_min, longitude_max, latitude_max

That matches OpenAQ’s bbox format: min X, min Y, max X, max Y.

In [ ]:
GULF_CITIES = [
    {
        "city": "Dubai",
        "country": "United Arab Emirates",
        "iso": "AE",
        "bbox": [54.85, 24.80, 55.70, 25.45],
        "group": "Gulf / desert urbanization",
    },
    {
        "city": "Abu Dhabi",
        "country": "United Arab Emirates",
        "iso": "AE",
        "bbox": [54.10, 24.20, 54.85, 24.70],
        "group": "Gulf / desert urbanization",
    },
    {
        "city": "Riyadh",
        "country": "Saudi Arabia",
        "iso": "SA",
        "bbox": [46.35, 24.35, 47.10, 25.05],
        "group": "Gulf / desert urbanization",
    },
    {
        "city": "Doha",
        "country": "Qatar",
        "iso": "QA",
        "bbox": [51.35, 25.15, 51.75, 25.45],
        "group": "Gulf / desert urbanization",
    },
    {
        "city": "Kuwait City",
        "country": "Kuwait",
        "iso": "KW",
        "bbox": [47.75, 29.15, 48.20, 29.50],
        "group": "Gulf / desert urbanization",
    },
]

#### Section 3: API helper functions

In [ ]:
def openaq_get(endpoint, params=None, max_retries=3, sleep_seconds=2):
    # General OpenAQ GET request helper
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    params = params or {}

    for attempt in range(max_retries):
        response = requests.get(url, headers=HEADERS, params=params, timeout=60)

        if response.status_code == 429 and attempt < max_retries - 1:
            time.sleep(sleep_seconds * (attempt + 1))
            continue

        response.raise_for_status()
        return response.json()

    response.raise_for_status()


def fetch_paginated(endpoint, params=None, limit=1000, pause_seconds=0.2):
    # Pagination helper for OpenAQ list endpoints
    params = params.copy() if params else {}
    params["limit"] = limit

    page = 1
    results = []

    while True:
        params["page"] = page
        payload = openaq_get(endpoint, params=params)

        batch = payload.get("results", [])
        meta = payload.get("meta", {})
        found = meta.get("found")

        results.extend(batch)

        if len(batch) == 0:
            break

        if found is not None and len(results) >= found:
            break

        page += 1
        time.sleep(pause_seconds)

    return results

#### Section 4: Resolve PM2.5 and PM10 parameter IDs

We did not hardcode pm10, because OpenAQ parameter IDs can be checked directly from the API.

In [ ]:
# Parameter metadata from OpenAQ
parameters = fetch_paginated("parameters", params={}, limit=1000)

parameters_df = pd.json_normalize(parameters)

parameters_df[parameters_df["name"].isin(["pm25", "pm10"])][
    ["id", "name", "displayName", "units", "description"]
]

parameter_lookup = parameters_df.set_index("name")["id"].to_dict()

PM25_ID = parameter_lookup["pm25"]
PM10_ID = parameter_lookup["pm10"]

PM25_ID, PM10_ID

#### Section 5: Sensor parsing helpers

In [ ]:
def parse_openaq_datetime(value):
    # Datetime parser for OpenAQ nested datetime fields
    if value is None:
        return pd.NaT

    if isinstance(value, dict):
        value = value.get("utc") or value.get("local")

    return pd.to_datetime(value, utc=True, errors="coerce")


def get_nested_name(value):
    # Name extractor for nested metadata fields
    if isinstance(value, dict):
        return value.get("name")
    return None


def summarize_sensor_dates(sensors, parameter_name):
    # Sensor date summary for one parameter at one location
    matching_sensors = []

    for sensor in sensors:
        parameter = sensor.get("parameter", {})
        if parameter.get("name") == parameter_name:
            matching_sensors.append(sensor)

    if len(matching_sensors) == 0:
        return {"sensor_ids": [], "first": pd.NaT, "last": pd.NaT, "sensor_count": 0}

    first_dates = [
        parse_openaq_datetime(sensor.get("datetimeFirst"))
        for sensor in matching_sensors
    ]

    last_dates = [
        parse_openaq_datetime(sensor.get("datetimeLast")) for sensor in matching_sensors
    ]

    first_dates = [date for date in first_dates if pd.notna(date)]
    last_dates = [date for date in last_dates if pd.notna(date)]

    return {
        "sensor_ids": [sensor.get("id") for sensor in matching_sensors],
        "first": min(first_dates) if first_dates else pd.NaT,
        "last": max(last_dates) if last_dates else pd.NaT,
        "sensor_count": len(matching_sensors),
    }

#### Section 6: Convert OpenAQ locations into audit rows

In [ ]:
def location_to_audit_row(location, city_spec):
    # Location metadata converted into one audit row
    sensors = location.get("sensors", [])

    pm25_summary = summarize_sensor_dates(sensors, "pm25")
    pm10_summary = summarize_sensor_dates(sensors, "pm10")

    has_pm25 = pm25_summary["sensor_count"] > 0
    has_pm10 = pm10_summary["sensor_count"] > 0

    overlap_start = pd.NaT
    overlap_end = pd.NaT
    overlap_months = 0

    if has_pm25 and has_pm10:
        overlap_start = max(pm25_summary["first"], pm10_summary["first"])
        overlap_end = min(pm25_summary["last"], pm10_summary["last"])

        if (
            pd.notna(overlap_start)
            and pd.notna(overlap_end)
            and overlap_end > overlap_start
        ):
            overlap_months = (overlap_end - overlap_start).days / 30.44

    coordinates = location.get("coordinates") or {}

    return {
        "city": city_spec["city"],
        "country": city_spec["country"],
        "iso": city_spec["iso"],
        "city_group": city_spec["group"],
        "location_id": location.get("id"),
        "location_name": location.get("name"),
        "latitude": coordinates.get("latitude"),
        "longitude": coordinates.get("longitude"),
        "timezone": location.get("timezone"),
        "provider": get_nested_name(location.get("provider")),
        "owner": get_nested_name(location.get("owner")),
        "is_mobile": location.get("isMobile"),
        "is_monitor": location.get("isMonitor"),
        "has_pm25": has_pm25,
        "has_pm10": has_pm10,
        "pm25_sensor_count": pm25_summary["sensor_count"],
        "pm10_sensor_count": pm10_summary["sensor_count"],
        "pm25_sensor_ids": pm25_summary["sensor_ids"],
        "pm10_sensor_ids": pm10_summary["sensor_ids"],
        "pm25_first": pm25_summary["first"],
        "pm25_last": pm25_summary["last"],
        "pm10_first": pm10_summary["first"],
        "pm10_last": pm10_summary["last"],
        "overlap_start": overlap_start,
        "overlap_end": overlap_end,
        "overlap_months": overlap_months,
        "s3_prefix": f"s3://openaq-data-archive/records/csv.gz/locationid={location.get('id')}/",
    }

#### Section 7: Fetch Gulf locations

In [ ]:
def fetch_locations_for_city(city_spec):
    # Location query for one city bounding box
    bbox_string = ",".join(str(value) for value in city_spec["bbox"])

    params = {
        "bbox": bbox_string,
        "iso": city_spec["iso"],
        "parameters_id": f"{PM25_ID},{PM10_ID}",
        "mobile": "false",
        "limit": 1000,
    }

    locations = fetch_paginated("locations", params=params, limit=1000)

    rows = [location_to_audit_row(location, city_spec) for location in locations]

    return rows

In [ ]:
# Gulf city location audit
gulf_rows = []

for city_spec in GULF_CITIES:
    city_rows = fetch_locations_for_city(city_spec)
    gulf_rows.extend(city_rows)

gulf_audit = pd.DataFrame(gulf_rows)

gulf_audit.shape

In [ ]:
gulf_audit.head()

#### Section 8: Keep only locations with both PM2.5 and PM10

In [ ]:
gulf_pm_locations = gulf_audit[
    (gulf_audit["has_pm25"]) & (gulf_audit["has_pm10"])
].copy()

In [ ]:
gulf_pm_locations["candidate_keep_24_months"] = (
    gulf_pm_locations["overlap_months"] >= 24
)

In [ ]:
gulf_pm_locations = gulf_pm_locations.sort_values(
    by=["city", "candidate_keep_24_months", "overlap_months"],
    ascending=[True, False, False],
)

In [ ]:
display_columns = [
    "city",
    "location_id",
    "location_name",
    "latitude",
    "longitude",
    "provider",
    "is_monitor",
    "pm25_sensor_count",
    "pm10_sensor_count",
    "pm25_sensor_ids",
    "pm10_sensor_ids",
    "overlap_start",
    "overlap_end",
    "overlap_months",
    "candidate_keep_24_months",
    "s3_prefix",
]

gulf_pm_locations[display_columns]

#### Section 9: Summary by city

In [ ]:
city_summary = (
    gulf_pm_locations.groupby("city")
    .agg(
        locations_with_both_pm=("location_id", "nunique"),
        candidate_locations_24_months=("candidate_keep_24_months", "sum"),
        max_overlap_months=("overlap_months", "max"),
        median_overlap_months=("overlap_months", "median"),
    )
    .reset_index()
    .sort_values("locations_with_both_pm", ascending=False)
)

city_summary

#### Section 10: Save Gulf audit results

In [ ]:
output_dir = Path("data/metadata/gulf")
output_dir.mkdir(parents=True, exist_ok=True)

gulf_audit.to_csv(output_dir / "gulf_all_openaq_locations.csv", index=False)

gulf_pm_locations.to_csv(output_dir / "gulf_locations_with_pm25_pm10.csv", index=False)

gulf_pm_locations.to_json(
    output_dir / "gulf_locations_with_pm25_pm10.json",
    orient="records",
    indent=2,
    date_format="iso",
)

#### Section 11: Clean location ID list for S3 step

In [ ]:
gulf_location_ids = gulf_pm_locations[
    [
        "city",
        "location_id",
        "location_name",
        "pm25_sensor_ids",
        "pm10_sensor_ids",
        "overlap_start",
        "overlap_end",
        "overlap_months",
        "s3_prefix",
    ]
].copy()

gulf_location_ids

In [ ]:
gulf_location_ids.to_csv(
    output_dir / "gulf_location_ids_for_s3_download.csv", index=False
)

#### Section 12: pollutant availability tiers

The initial strict table keeps only locations where PM2.5 and PM10 are available at the same location. This table keeps the project scalable by separating locations into PM2.5 core locations, PM10 context locations, and paired PM2.5 plus PM10 locations.


In [ ]:
gulf_audit_revised = gulf_audit.copy()

# Pollutant availability categories for scalable city selection
gulf_audit_revised["eligible_core_pm25"] = gulf_audit_revised["has_pm25"]
gulf_audit_revised["eligible_pm10_context"] = gulf_audit_revised["has_pm10"]
gulf_audit_revised["eligible_particle_profile"] = (
    gulf_audit_revised["has_pm25"] & gulf_audit_revised["has_pm10"]
)

gulf_audit_revised["coverage_tier"] = "Exclude"

gulf_audit_revised.loc[
    gulf_audit_revised["eligible_core_pm25"]
    & gulf_audit_revised["eligible_particle_profile"],
    "coverage_tier",
] = "Tier A - PM2.5 and PM10"

gulf_audit_revised.loc[
    gulf_audit_revised["eligible_core_pm25"]
    & ~gulf_audit_revised["eligible_particle_profile"],
    "coverage_tier",
] = "Tier B - PM2.5 only"

gulf_audit_revised.loc[
    ~gulf_audit_revised["eligible_core_pm25"]
    & gulf_audit_revised["eligible_pm10_context"],
    "coverage_tier",
] = "Tier C - PM10 only"


In [ ]:
revised_display_columns = [
    "city",
    "location_id",
    "location_name",
    "latitude",
    "longitude",
    "provider",
    "is_monitor",
    "has_pm25",
    "has_pm10",
    "pm25_sensor_count",
    "pm10_sensor_count",
    "pm25_sensor_ids",
    "pm10_sensor_ids",
    "coverage_tier",
    "eligible_core_pm25",
    "eligible_pm10_context",
    "eligible_particle_profile",
    "s3_prefix",
]

gulf_audit_revised[revised_display_columns].sort_values(
    by=["city", "coverage_tier", "location_id"]
)


#### Section 13: City-level pollutant availability summary

This summary is for deciding which Gulf cities should continue into the S3 coverage check. PM2.5 is the core project pollutant. PM10 is used where available for particle-profile and episode-type analysis.


In [ ]:
city_pollutant_summary = (
    gulf_audit_revised.groupby("city")
    .agg(
        total_locations=("location_id", "nunique"),
        pm25_locations=("has_pm25", "sum"),
        pm10_locations=("has_pm10", "sum"),
        paired_locations=("eligible_particle_profile", "sum"),
        tier_a_locations=(
            "coverage_tier",
            lambda values: (values == "Tier A - PM2.5 and PM10").sum(),
        ),
        tier_b_locations=(
            "coverage_tier",
            lambda values: (values == "Tier B - PM2.5 only").sum(),
        ),
        tier_c_locations=(
            "coverage_tier",
            lambda values: (values == "Tier C - PM10 only").sum(),
        ),
    )
    .reset_index()
)

city_pollutant_summary["city_has_pm25"] = city_pollutant_summary["pm25_locations"] > 0
city_pollutant_summary["city_has_pm10"] = city_pollutant_summary["pm10_locations"] > 0
city_pollutant_summary["city_has_both"] = (
    city_pollutant_summary["city_has_pm25"]
    & city_pollutant_summary["city_has_pm10"]
)

city_pollutant_summary["recommended_action"] = "Exclude for now"

city_pollutant_summary.loc[
    city_pollutant_summary["city_has_both"],
    "recommended_action",
] = "Keep for full S3 coverage check"

city_pollutant_summary.loc[
    city_pollutant_summary["city_has_pm25"]
    & ~city_pollutant_summary["city_has_pm10"],
    "recommended_action",
] = "Keep for PM2.5 core; exclude from ratio analysis"

city_pollutant_summary.loc[
    ~city_pollutant_summary["city_has_pm25"]
    & city_pollutant_summary["city_has_pm10"],
    "recommended_action",
] = "Optional PM10 context only"

city_pollutant_summary.sort_values(
    by=["city_has_pm25", "city_has_both", "pm25_locations"],
    ascending=[False, False, False],
)


#### Section 14: Revised location ID lists for S3 coverage check

The paired list is useful for PM2.5 / PM10 ratio analysis. The PM2.5 core list is useful for the main global comparison, exceedance burden, seasonality, rankings, and maps.


In [ ]:
gulf_pm25_core_locations = (
    gulf_audit_revised[gulf_audit_revised["eligible_core_pm25"]]
    .sort_values(by=["city", "location_id"])
    .copy()
)

gulf_pm10_context_locations = (
    gulf_audit_revised[gulf_audit_revised["eligible_pm10_context"]]
    .sort_values(by=["city", "location_id"])
    .copy()
)

gulf_particle_profile_locations = (
    gulf_audit_revised[gulf_audit_revised["eligible_particle_profile"]]
    .sort_values(by=["city", "location_id"])
    .copy()
)


In [ ]:
s3_columns = [
    "city",
    "location_id",
    "location_name",
    "has_pm25",
    "has_pm10",
    "pm25_sensor_ids",
    "pm10_sensor_ids",
    "coverage_tier",
    "s3_prefix",
]

gulf_pm25_core_locations[s3_columns]


In [ ]:
# Revised Gulf metadata exports
gulf_audit_revised.to_csv(
    output_dir / "gulf_all_openaq_locations_with_tiers.csv",
    index=False,
)

city_pollutant_summary.to_csv(
    output_dir / "gulf_city_pollutant_summary.csv",
    index=False,
)

gulf_pm25_core_locations[s3_columns].to_csv(
    output_dir / "gulf_pm25_core_location_ids_for_s3_download.csv",
    index=False,
)

gulf_pm10_context_locations[s3_columns].to_csv(
    output_dir / "gulf_pm10_context_location_ids_for_s3_download.csv",
    index=False,
)

gulf_particle_profile_locations[s3_columns].to_csv(
    output_dir / "gulf_particle_profile_location_ids_for_s3_download.csv",
    index=False,
)


#### Section 15: Later city group sections

Additional candidate city groups can be added below this point using the same functions and the same tiered eligibility logic.

Planned groups:

- High-pollution / fast-growing Asia
- Dense but policy-relevant Asian comparison
- Policy / developed-city comparison
- Optional contrast cities


In [ ]:
fetch_locations_for_city
GULF-style audit functions
PM25_ID
PM10_ID

16 

In [ ]:
# High-pollution / fast-growing Asia
ASIA_CITIES = [
    {
        "city": "Delhi",
        "country": "India",
        "iso": "IN",
        "bbox": [76.80, 28.35, 77.45, 28.95],
        "group": "High-pollution / fast-growing Asia",
    },
    {
        "city": "Lahore",
        "country": "Pakistan",
        "iso": "PK",
        "bbox": [74.05, 31.25, 74.60, 31.75],
        "group": "High-pollution / fast-growing Asia",
    },
    {
        "city": "Dhaka",
        "country": "Bangladesh",
        "iso": "BD",
        "bbox": [90.25, 23.60, 90.60, 24.00],
        "group": "High-pollution / fast-growing Asia",
    },
    {
        "city": "Bangkok",
        "country": "Thailand",
        "iso": "TH",
        "bbox": [100.30, 13.50, 100.95, 14.15],
        "group": "High-pollution / fast-growing Asia",
    },
    {
        "city": "Jakarta",
        "country": "Indonesia",
        "iso": "ID",
        "bbox": [106.60, -6.45, 107.05, -5.95],
        "group": "High-pollution / fast-growing Asia",
    },
]

In [ ]:
# Asia city location audit
asia_rows = []

for city_spec in ASIA_CITIES:
    city_rows = fetch_locations_for_city(city_spec)
    asia_rows.extend(city_rows)

asia_audit = pd.DataFrame(asia_rows)

asia_audit.shape

In [ ]:
asia_audit.head()

#### # Section 17: Asia locations with both PM2.5 and PM10

In [ ]:
asia_pm_locations = asia_audit[
    (asia_audit["has_pm25"]) & (asia_audit["has_pm10"])
].copy()

asia_pm_locations["candidate_keep_24_months"] = (
    asia_pm_locations["overlap_months"] >= 24
)

In [ ]:
asia_pm_locations = asia_pm_locations.sort_values(
    by=["city", "candidate_keep_24_months", "overlap_months"],
    ascending=[True, False, False],
)

In [ ]:
display_columns = [
    "city",
    "location_id",
    "location_name",
    "latitude",
    "longitude",
    "provider",
    "is_monitor",
    "pm25_sensor_count",
    "pm10_sensor_count",
    "pm25_sensor_ids",
    "pm10_sensor_ids",
    "overlap_start",
    "overlap_end",
    "overlap_months",
    "candidate_keep_24_months",
    "s3_prefix",
]

asia_pm_locations[display_columns]

#### # Section 18: Asia summary by city

In [ ]:
asia_city_summary = (
    asia_pm_locations.groupby("city")
    .agg(
        locations_with_both_pm=("location_id", "nunique"),
        candidate_locations_24_months=("candidate_keep_24_months", "sum"),
        max_overlap_months=("overlap_months", "max"),
        median_overlap_months=("overlap_months", "median"),
    )
    .reset_index()
    .sort_values("locations_with_both_pm", ascending=False)
)

asia_city_summary

#### # Section 19: Save Asia audit results

In [ ]:
asia_output_dir = Path("data/metadata/asia")
asia_output_dir.mkdir(parents=True, exist_ok=True)

asia_audit.to_csv(asia_output_dir / "asia_all_openaq_locations.csv", index=False)

asia_pm_locations.to_csv(
    asia_output_dir / "asia_locations_with_pm25_pm10.csv", index=False
)

asia_pm_locations.to_json(
    asia_output_dir / "asia_locations_with_pm25_pm10.json",
    orient="records",
    indent=2,
    date_format="iso",
)

#### # Section 20: Clean Asia location ID list for S3 step

In [ ]:
asia_location_ids = asia_pm_locations[
    [
        "city",
        "location_id",
        "location_name",
        "pm25_sensor_ids",
        "pm10_sensor_ids",
        "overlap_start",
        "overlap_end",
        "overlap_months",
        "s3_prefix",
    ]
].copy()

asia_location_ids

In [ ]:
asia_location_ids.to_csv(
    asia_output_dir / "asia_location_ids_for_s3_download.csv", index=False
)

In [ ]:
asia_audit_with_tiers = asia_audit.copy()

# City-location eligibility categories
asia_audit_with_tiers["eligible_core_pm25"] = asia_audit_with_tiers["has_pm25"]

asia_audit_with_tiers["eligible_pm10_context"] = asia_audit_with_tiers["has_pm10"]

asia_audit_with_tiers["eligible_particle_profile"] = (
    asia_audit_with_tiers["has_pm25"] & asia_audit_with_tiers["has_pm10"]
)

asia_audit_with_tiers["coverage_tier"] = "Exclude"

asia_audit_with_tiers.loc[
    asia_audit_with_tiers["eligible_core_pm25"]
    & asia_audit_with_tiers["eligible_particle_profile"],
    "coverage_tier",
] = "Tier A - PM2.5 and PM10"

asia_audit_with_tiers.loc[
    asia_audit_with_tiers["eligible_core_pm25"]
    & ~asia_audit_with_tiers["eligible_particle_profile"],
    "coverage_tier",
] = "Tier B - PM2.5 only"

asia_audit_with_tiers.loc[
    ~asia_audit_with_tiers["eligible_core_pm25"]
    & asia_audit_with_tiers["eligible_pm10_context"],
    "coverage_tier",
] = "Tier C - PM10 only"



#### # Section 22: Asia city-level pollutant availability summary

In [ ]:
asia_city_pollutant_summary = (
    asia_audit_with_tiers.groupby("city")
    .agg(
        total_locations=("location_id", "nunique"),
        pm25_locations=("has_pm25", "sum"),
        pm10_locations=("has_pm10", "sum"),
        paired_locations=("eligible_particle_profile", "sum"),
        tier_a_locations=(
            "coverage_tier",
            lambda x: (x == "Tier A - PM2.5 and PM10").sum(),
        ),
        tier_b_locations=(
            "coverage_tier",
            lambda x: (x == "Tier B - PM2.5 only").sum(),
        ),
        tier_c_locations=("coverage_tier", lambda x: (x == "Tier C - PM10 only").sum()),
    )
    .reset_index()
)

asia_city_pollutant_summary["city_has_pm25"] = (
    asia_city_pollutant_summary["pm25_locations"] > 0
)

asia_city_pollutant_summary["city_has_pm10"] = (
    asia_city_pollutant_summary["pm10_locations"] > 0
)

asia_city_pollutant_summary["city_has_both"] = (
    asia_city_pollutant_summary["city_has_pm25"]
    & asia_city_pollutant_summary["city_has_pm10"]
)

asia_city_pollutant_summary["recommended_action"] = "Exclude for now"

asia_city_pollutant_summary.loc[
    asia_city_pollutant_summary["city_has_both"], "recommended_action"
] = "Keep for full audit"

asia_city_pollutant_summary.loc[
    asia_city_pollutant_summary["city_has_pm25"]
    & ~asia_city_pollutant_summary["city_has_pm10"],
    "recommended_action",
] = "Keep for PM2.5 core, exclude from ratio analysis"

asia_city_pollutant_summary.loc[
    ~asia_city_pollutant_summary["city_has_pm25"]
    & asia_city_pollutant_summary["city_has_pm10"],
    "recommended_action",
] = "Optional PM10 context only"

asia_city_pollutant_summary.sort_values(
    ["city_has_both", "pm25_locations", "paired_locations"],
    ascending=[False, False, False],
)

#### # Section 23: Asia S3 location ID lists

In [ ]:
asia_pm25_core_locations = asia_audit_with_tiers[
    asia_audit_with_tiers["eligible_core_pm25"]
].copy()

asia_pm10_context_locations = asia_audit_with_tiers[
    asia_audit_with_tiers["eligible_pm10_context"]
].copy()

asia_particle_profile_locations = asia_audit_with_tiers[
    asia_audit_with_tiers["eligible_particle_profile"]
].copy()

asia_pm25_core_ids = (
    asia_pm25_core_locations[
        [
            "city",
            "location_id",
            "location_name",
            "coverage_tier",
            "pm25_sensor_ids",
            "pm10_sensor_ids",
            "s3_prefix",
        ]
    ]
    .sort_values(["city", "coverage_tier", "location_name"])
    .copy()
)

asia_pm10_context_ids = (
    asia_pm10_context_locations[
        [
            "city",
            "location_id",
            "location_name",
            "coverage_tier",
            "pm25_sensor_ids",
            "pm10_sensor_ids",
            "s3_prefix",
        ]
    ]
    .sort_values(["city", "coverage_tier", "location_name"])
    .copy()
)

asia_particle_profile_ids = (
    asia_particle_profile_locations[
        [
            "city",
            "location_id",
            "location_name",
            "coverage_tier",
            "pm25_sensor_ids",
            "pm10_sensor_ids",
            "s3_prefix",
        ]
    ]
    .sort_values(["city", "coverage_tier", "location_name"])
    .copy()
)



In [ ]:
asia_pm25_core_ids

In [ ]:
asia_particle_profile_ids

In [ ]:
asia_audit_with_tiers.to_csv(
    asia_output_dir / "asia_all_openaq_locations_with_tiers.csv", index=False
)

asia_city_pollutant_summary.to_csv(
    asia_output_dir / "asia_city_pollutant_summary.csv", index=False
)

asia_pm25_core_ids.to_csv(
    asia_output_dir / "asia_pm25_core_location_ids_for_s3_download.csv", index=False
)

asia_pm10_context_ids.to_csv(
    asia_output_dir / "asia_pm10_context_location_ids_for_s3_download.csv", index=False
)

asia_particle_profile_ids.to_csv(
    asia_output_dir / "asia_particle_profile_location_ids_for_s3_download.csv",
    index=False,
)

#### Fetching all other city location

In [ ]:
from openaq_city_audit_helper import (
    audit_multiple_city_groups,
    combine_group_outputs,
)

In [ ]:
results = audit_multiple_city_groups(
    group_keys=["dense_policy_asia", "developed_city", "optional_contrast"],
    api_key=OPENAQ_API_KEY,
    output_root="data/metadata",
)

combined_results = combine_group_outputs(results=results, output_root="data/metadata")

In [ ]:
results["dense_policy_asia"]["city_pollutant_summary"]

In [ ]:
results["developed_city"]["particle_profile_ids"]